In [1]:
import cv2
import numpy as np
import os
import pandas as pd
from pathlib import Path
from skimage import exposure, filters, morphology
from PIL import Image
import matplotlib.pyplot as plt

# ============================================
# 1. Configuration et chemins
# ============================================

In [2]:

from pathlib import Path

# Chemin du dossier contenant ce script (robuste)
#SCRIPT_DIR = Path.cwd()  # ou Path("/media/jf/.../jul26_bds_blood_cells")
#ROOT = SCRIPT_DIR.parent

ROOT = Path.cwd().parent

# Datasets
LMU_DIR = ROOT / "data" / "PKG-AML-Cytomorphology_LMU"
PBC_DIR = ROOT / "data" / "PBC_dataset_normal_DIB" / "PBC_dataset_normal_DIB"

# Fichiers d'annotations pour LMU
ANNOTATIONS_FILE = LMU_DIR / "annotations.dat"
SUMS_FILE = LMU_DIR / "AML-Cytomorphology.sums"

# Dossiers de sortie
OUTPUT_LMU_DIR = ROOT / "output" / "processed_LMU"
OUTPUT_PBC_DIR = ROOT / "output" / "processed_PBC"

# Créer les dossiers de sortie s'ils n'existent pas
os.makedirs(OUTPUT_LMU_DIR, exist_ok=True)
os.makedirs(OUTPUT_PBC_DIR, exist_ok=True)

In [3]:
print("ROOT:", ROOT)
print("LMU_DIR:", LMU_DIR)
print("ANNOTATIONS_FILE:", ANNOTATIONS_FILE)
print("Existe ?", ANNOTATIONS_FILE.exists())

ROOT: /media/jf/4C267D63267D4EC8/Liora/Blood_cells_classification/Blood_cells_classification/jul26_bds_blood_cells
LMU_DIR: /media/jf/4C267D63267D4EC8/Liora/Blood_cells_classification/Blood_cells_classification/jul26_bds_blood_cells/data/PKG-AML-Cytomorphology_LMU
ANNOTATIONS_FILE: /media/jf/4C267D63267D4EC8/Liora/Blood_cells_classification/Blood_cells_classification/jul26_bds_blood_cells/data/PKG-AML-Cytomorphology_LMU/annotations.dat
Existe ? True


# ============================================
# 2. Fonctions de préprocessing
# ============================================

In [4]:
def load_image(image_path):
    """Charger une image en niveaux de gris."""
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Impossible de charger l'image : {image_path}")
    return image

In [5]:
def denoise_image(image, kernel_size=5):
    """Appliquer un filtre de débruitage (médiane ou gaussien)."""
    # Filtre médian pour supprimer le bruit
    denoised = cv2.medianBlur(image, kernel_size)
    return denoised

In [6]:
def enhance_contrast(image, clip_limit=0.03, grid_size=(8, 8)):
    """Améliorer le contraste avec CLAHE (Contrast Limited Adaptive Histogram Equalization)."""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    enhanced = clahe.apply(image)
    return enhanced

In [7]:
def normalize_image(image):
    """Normaliser l'image entre 0 et 255."""
    normalized = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)
    return normalized.astype(np.uint8)

In [8]:
def threshold_image(image, method="otsu"):
    """Appliquer un seuillage pour segmenter les cellules."""
    if method == "otsu":
        _, thresh = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        # Seuillage adaptatif
        thresh = cv2.adaptiveThreshold(
            image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
        )
    return thresh

In [9]:
def remove_small_objects(image, min_size=50):
    """Supprimer les petits objets (bruit résiduel)."""
    # Trouver les contours
    contours, _ = cv2.findContours(image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Créer un masque vide
    mask = np.zeros_like(image)
    
    # Dessiner uniquement les contours suffisamment grands
    for contour in contours:
        if cv2.contourArea(contour) >= min_size:
            cv2.drawContours(mask, [contour], -1, 255, thickness=cv2.FILLED)
    
    return mask

In [10]:
def resize_image(image, target_size=(512, 512)):
    """Redimensionner l'image."""
    resized = cv2.resize(image, target_size, interpolation=cv2.INTER_AREA)
    return resized

In [11]:
def save_image(image, output_path):
    """Sauvegarder l'image."""
    cv2.imwrite(output_path, image)

In [12]:
def apply_morphological_operations(image, kernel_size=3, operation="opening"):
    """Appliquer des opérations morphologiques (ouverture, fermeture)."""
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    
    if operation == "opening":
        processed = cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel)
    elif operation == "closing":
        processed = cv2.morphologyEx(image, cv2.MORPH_CLOSE, kernel)
    else:
        processed = image
    
    return processed

In [13]:
def preprocess_image(image_path, output_path):
    """Appliquer le préprocessing complet à une image."""
    # 1. Charger l'image
    image = load_image(image_path)
    
    # 2. Débruiter
    denoised = denoise_image(image, kernel_size=3)
    
    # 3. Améliorer le contraste
    enhanced = enhance_contrast(denoised, clip_limit=0.03)
    
    # 4. Normaliser
    normalized = normalize_image(enhanced)
    
    # 5. Seuillage
    thresh = threshold_image(normalized, method="otsu")
    
    # 6. Opérations morphologiques (ouverture pour supprimer le bruit)
    morph = apply_morphological_operations(thresh, kernel_size=3, operation="opening")
    
    # 7. Supprimer les petits objets
    cleaned = remove_small_objects(morph, min_size=100)
    
    # 8. Sauvegarder
    save_image(cleaned, output_path)
    
    return cleaned

# ============================================
# 3. Lire les données des datasets
# ============================================

In [14]:
def get_LMU_image_paths():
    """Extraire les chemins des images pour le dataset LMU."""
    # Lire les annotations (format : relative_path class_code re-annotation re-re-annotation)
    df_pkg_annotation = pd.read_csv(
        ANNOTATIONS_FILE, sep=" ", header=None,
        names=["relative_path", "class_code", "re_annotation", "re_re_annotation"]
    )
    
    # Extraire les chemins absolus des images (les chemins dans annotations.dat sont déjà relatifs à LMU_DIR)
    image_paths = [LMU_DIR / path for path in df_pkg_annotation["relative_path"]]
    
    # Filtrer pour ne garder que les fichiers existants (au cas où)
    image_paths = [path for path in image_paths if path.exists()]
    return image_paths

In [15]:
def get_PBC_image_paths():
    """Extraire les chemins des images pour le dataset PBC."""
    image_paths = []
    for root, _, files in os.walk(PBC_DIR):
        for file in files:
            if file.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff")):
                image_paths.append(Path(root) / file)
    return image_paths

# ============================================
# 4. Traiter toutes les images d'un dataset
# ============================================

In [16]:
def process_dataset(image_paths, output_dir, dataset_name):
    """Traiter toutes les images d'une liste de chemins."""
    for image_path in image_paths:
        try:
            # Vérifier si le fichier existe
            if not image_path.exists():
                print(f"⚠️ Fichier introuvable : {image_path}")
                continue
            
            # Créer le chemin de sortie
            output_path = output_dir / f"processed_{image_path.stem}{image_path.suffix}"
            
            print(f"Traitement de {image_path.name} ({dataset_name})...")
            preprocess_image(str(image_path), str(output_path))
            print(f"✅ {image_path.name} traité et sauvegardé sous {output_path}")
        except Exception as e:
            print(f"❌ Erreur lors du traitement de {image_path.name}: {e}")

# ============================================
# 4. Visualisation (optionnel)
# ============================================

In [17]:
def visualize_results(input_path, output_path):
    """Visualiser l'image originale et traitée côte à côte."""
    input_img = load_image(input_path)
    output_img = load_image(output_path)
    
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    plt.title("Image originale")
    plt.imshow(input_img, cmap="gray")
    plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.title("Image traitée")
    plt.imshow(output_img, cmap="gray")
    plt.axis("off")
    
    plt.tight_layout()
    plt.show()


# ============================================
# 5. Exécution
# ============================================

In [18]:
if __name__ == "__main__":
    # Lire les chemins des images pour chaque dataset
    print("🔍 Lecture des chemins des images...")
    lmu_paths = get_LMU_image_paths()
    pbc_paths = get_PBC_image_paths()
    
    print(f"📌 Nombre d'images LMU : {len(lmu_paths)}")
    print(f"📌 Nombre d'images PBC : {len(pbc_paths)}")
    
    # Traiter les images du dataset LMU
    print("\n🚀 Début du traitement pour LMU...")
    process_dataset(lmu_paths, OUTPUT_LMU_DIR, "LMU")
    
    # Traiter les images du dataset PBC
    print("\n🚀 Début du traitement pour PBC...")
    process_dataset(pbc_paths, OUTPUT_PBC_DIR, "PBC")
    
    print("\n✅ Traitement terminé pour les deux datasets !")
    
    # Exemple de visualisation (décommenter pour tester)
    # if lmu_paths:
    #     visualize_results(str(lmu_paths[0]), str(OUTPUT_LMU_DIR / f"processed_{Path(lmu_paths[0]).stem}{Path(lmu_paths[0]).suffix}"))

🔍 Lecture des chemins des images...
📌 Nombre d'images LMU : 18365
📌 Nombre d'images PBC : 17093

🚀 Début du traitement pour LMU...
Traitement de BAS_0001.tiff (LMU)...
✅ BAS_0001.tiff traité et sauvegardé sous /media/jf/4C267D63267D4EC8/Liora/Blood_cells_classification/Blood_cells_classification/jul26_bds_blood_cells/output/processed_LMU/processed_BAS_0001.tiff
Traitement de BAS_0002.tiff (LMU)...
✅ BAS_0002.tiff traité et sauvegardé sous /media/jf/4C267D63267D4EC8/Liora/Blood_cells_classification/Blood_cells_classification/jul26_bds_blood_cells/output/processed_LMU/processed_BAS_0002.tiff
Traitement de BAS_0003.tiff (LMU)...
✅ BAS_0003.tiff traité et sauvegardé sous /media/jf/4C267D63267D4EC8/Liora/Blood_cells_classification/Blood_cells_classification/jul26_bds_blood_cells/output/processed_LMU/processed_BAS_0003.tiff
Traitement de BAS_0004.tiff (LMU)...
✅ BAS_0004.tiff traité et sauvegardé sous /media/jf/4C267D63267D4EC8/Liora/Blood_cells_classification/Blood_cells_classification/jul2